In [ ]:
import pandas as pd
import torch
from rdkit import Chem
from torch_geometric.data import Data
print(torch.__version__)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

In [ ]:
data = pd.read_csv("../Dataset/qm8.csv")

In [ ]:
data = data.iloc[:100,:]

In [ ]:
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Add hydrogens explicitly as they are often important in QM datasets
    mol = Chem.AddHs(mol)
    
    # Atom features: Atomic number
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([atom.GetAtomicNum()])
    x = torch.tensor(atom_features, dtype=torch.float)
    
    # Edge index: Bonds
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
    
    if not edge_index:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    return Data(x=x, edge_index=edge_index)

data['graph'] = data['smiles'].apply(smiles_to_graph)

In [ ]:
data.head()

In [ ]:
data.graph[1]

In [ ]:
G = to_networkx(data['graph'][0], to_undirected=True)

pos = nx.spring_layout(G, dim=3, seed=0)

node_xyz = np.array([pos[v] for v in sorted(G)])
edge_xyz = np.array([(pos[v], pos[u]) for u, v in G.edges()])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

for dim in (ax.xaxis, ax.yaxis, ax.zaxis):
    dim.set_ticks([])

ax.scatter(*node_xyz.T, s=500, c='#0A047A')

for vizedge in edge_xyz:
    ax.plot(*vizedge, color='k')

plt.show()